# 03 · Limpieza: Eventos Presenciales 2026_anonimizada

Encuesta de satisfacción posterior a los eventos presenciales de ACRIP Región Central (mar 2025 – sep 2026). Una fila = una respuesta.

- Entrada: `datos/originales/Eventos Presenciales 2026_anonimizada.xlsx` (hoja `Responses`)
- Salida: `datos/limpios/03_eventos_presenciales_limpio.csv`

In [ ]:
from pathlib import Path

import pandas as pd

BASE = Path.cwd().parent
ORIGINALES = BASE / "datos" / "originales"
LIMPIOS = BASE / "datos" / "limpios"
LIMPIOS.mkdir(exist_ok=True)

## 1. Carga
La fila 0 está vacía y los encabezados están en la fila 1, por eso `header=1`.

In [ ]:
df = pd.read_excel(ORIGINALES / "Eventos Presenciales 2026_anonimizada.xlsx", sheet_name="Responses", header=1)
assert df.shape == (407, 19), df.shape
df.head()

## 2. Renombrar y eliminar columnas
Cada columna se identifica por un fragmento de su nombre original (evita copiar a mano tildes, `→` y espacios especiales).
Al quedarse solo con las 14 columnas renombradas se eliminan: el consecutivo `Unnamed: 0`, las 3 columnas vaciadas al anonimizar (IP, teléfono, correo) y `Nombre del recopilador` (un solo valor, con el año equivocado).

In [ ]:
COLUMNAS = {
    "ID_RESPUESTA": "id_respuesta",
    "Respuesta iniciada": "fecha_inicio",
    "Respuesta completada": "fecha_fin",
    "locación": "cal_locacion",
    "convocatoria": "cal_convocatoria",
    "conferencistas": "cal_conferencistas",
    "temas tratados": "cal_temas",
    "temarios": "temas_sugeridos",
    "recibir actualizaciones": "recibir_info",
    "ID_PERSONA": "id_persona",
    "ID_EMPRESA": "id_empresa",
    "CIUDAD": "ciudad",
    "EMPRESA_EN_CRM": "empresa_en_crm",
    "METODO_ASIGNACION": "metodo_asignacion",
}
renombre = {col: nuevo for frag, nuevo in COLUMNAS.items() for col in df.columns if frag in col}
assert len(renombre) == len(COLUMNAS), renombre

df = df.rename(columns=renombre)[list(COLUMNAS.values())]
df.columns.tolist()

## 3. Fechas
Vienen como texto en español (`sep. 16, 2026 09:40:12 a. m.`) con un espacio especial (`\xa0`).
`pd.to_datetime` directo solo convierte 301 de 407 y sin avisar, así que se traduce el mes y el a. m./p. m. y se convierte con formato explícito.

In [ ]:
MESES = {"ene": "01", "feb": "02", "mar": "03", "abr": "04", "may": "05", "jun": "06",
         "jul": "07", "ago": "08", "sep": "09", "sept": "09", "oct": "10", "nov": "11", "dic": "12"}


def fecha_es(serie):
    texto = (serie.str.replace("\xa0", " ")
                  .str.replace(r"^([a-z]+)\.", lambda m: MESES[m.group(1)], regex=True)
                  .str.replace("a. m.", "AM").str.replace("p. m.", "PM"))
    return pd.to_datetime(texto, format="%m %d, %Y %I:%M:%S %p")


df["fecha_inicio"] = fecha_es(df["fecha_inicio"])
df["fecha_fin"] = fecha_es(df["fecha_fin"])
df["fecha_evento"] = df["fecha_inicio"].dt.normalize()
df["anio"] = df["fecha_inicio"].dt.year
df["mes"] = df["fecha_inicio"].dt.month

assert df["fecha_inicio"].notna().all() and df["fecha_fin"].notna().all()
assert (df["fecha_fin"] >= df["fecha_inicio"]).all()
df["anio"].value_counts()

## 4. Ciudad
16 variantes para 5 ciudades reales (tildes, D.C., errores de digitación). Se normaliza a minúsculas y se mapea a la forma oficial.
Si aparece una variante nueva, el `assert` falla y obliga a agregarla al diccionario. Vacíos → `Sin dato`.

In [ ]:
CIUDADES = {
    "bogotá": "Bogotá", "bogota": "Bogotá", "bogotá dc": "Bogotá", "bogotá d.c.": "Bogotá",
    "bogotá, d.c.": "Bogotá", "bogota d.c": "Bogotá",
    "medellín": "Medellín", "medellin": "Medellín", "medellib": "Medellín", "mediellín": "Medellín",
    "medelon": "Medellín", "mede": "Medellín",
    "envigado": "Envigado", "chia": "Chía", "itaguí": "Itagüí",
    "bogotá, armenia y cumaral": "Varias",
}
clave = df["ciudad"].str.strip().str.lower()
sin_mapear = set(clave.dropna()) - set(CIUDADES)
assert not sin_mapear, sin_mapear

df["ciudad"] = clave.map(CIUDADES).fillna("Sin dato")
df["ciudad"].value_counts()

## 5. Recibir información
"Otros - Ya la recibo / Ya estoy inscrita" significa que sí la quieren → `Sí`. "Otros" sin texto y vacíos → `Sin respuesta`.

In [ ]:
recibir = df["recibir_info"].str.strip()
df["recibir_info"] = (recibir.where(recibir.isin(["Sí", "No"]))
                             .mask(recibir.str.startswith("Otros (especifique) - Ya", na=False), "Sí")
                             .fillna("Sin respuesta"))
df["recibir_info"].value_counts()

## 6. Temas sugeridos
Se quitan espacios y se vacían las respuestas sin contenido (`.`, `No`, 1 carácter). `IA`/`Ia` se conservan (inteligencia artificial) unificadas como `IA`.

In [ ]:
temas = df["temas_sugeridos"].str.strip().replace({"Ia": "IA"})
df["temas_sugeridos"] = temas.mask(temas.isin([".", "No"]) | (temas.str.len() <= 1))
df["temas_sugeridos"].dropna().sort_values(key=lambda s: s.str.len()).head(10)

## 7. Anónimas y duplicados
- Las 80 respuestas sin persona/empresa/ciudad **se conservan**: sus calificaciones valen para los promedios. Se marcan con `anonima`.
- Duplicados: misma persona, mismo día y mismas 4 calificaciones. Se conserva la primera. Las anónimas no se comparan entre sí.

In [ ]:
CALIFICACIONES = ["cal_locacion", "cal_convocatoria", "cal_conferencistas", "cal_temas"]

df["anonima"] = df["id_persona"].isna()
duplicada = df.duplicated(["id_persona", "fecha_evento"] + CALIFICACIONES) & ~df["anonima"]
print("Eliminadas:", df.loc[duplicada, "id_respuesta"].tolist())

df = df[~duplicada].reset_index(drop=True)
assert len(df) == 405, len(df)

## 8. Validaciones y columnas finales

In [ ]:
assert df[CALIFICACIONES].isin([1, 2, 3, 4, 5]).all().all()
assert df["id_respuesta"].is_unique
assert df["id_empresa"].dropna().str.fullmatch(r"EMP-\d{4}").all()
assert df["id_persona"].dropna().str.fullmatch(r"P-\d{5}").all()

df["cal_promedio"] = df[CALIFICACIONES].mean(axis=1).round(2)
df["modalidad"] = "Presencial"

df = df[["id_respuesta", "fecha_inicio", "fecha_fin", "fecha_evento", "anio", "mes",
         *CALIFICACIONES, "cal_promedio", "temas_sugeridos", "recibir_info",
         "id_persona", "id_empresa", "ciudad", "empresa_en_crm", "metodo_asignacion",
         "anonima", "modalidad"]]
df.dtypes

## 9. Guardar
`utf-8-sig` para que Excel y Looker Studio lean bien las tildes.

In [ ]:
salida = LIMPIOS / "03_eventos_presenciales_limpio.csv"
df.to_csv(salida, index=False, encoding="utf-8-sig")

assert pd.read_csv(salida).shape == (405, 20)
salida.name